# Stage 5 – Building the three networks

**Goal:** group the partnership observations into one node table and one edge table per format, and prove that
the networks match the observations exactly.

**Gate:** all aggregation checks must pass, otherwise the notebook stops.

**Reads:** `data/processed/partnership_observations.csv`
**Writes:** `data/processed/nodes_<format>.csv`, `edges_<format>.csv`, `players.csv`,
`outputs/tables/validation_aggregation_checks.csv`

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_04_network_building.ipynb
stage_log = get_stage_logger("05_networks")

In [2]:
observations = read_table(PROCESSED_FOLDER / "partnership_observations.csv")
player_table = build_player_table(observations)
node_table = build_node_table(observations, player_table)
edge_table = build_edge_table(observations, player_table)
write_table(player_table, PROCESSED_FOLDER / "players.csv")

### Save one node and one edge table per format, and check each network

In [3]:
aggregation_results = []
for format_name in FORMATS:
    file_tag = FORMAT_FILE_TAG[format_name]
    write_table(node_table[node_table["format"] == format_name], PROCESSED_FOLDER / f"nodes_{file_tag}.csv")
    write_table(edge_table[edge_table["format"] == format_name], PROCESSED_FOLDER / f"edges_{file_tag}.csv")
    for check_name, passed, detail in aggregation_checks(observations, node_table, edge_table, format_name):
        aggregation_results.append({"format": format_name, "check": check_name, "passed": passed, "detail": detail})
    partnership_graph = build_format_graph(node_table, edge_table, format_name)
    stage_log.info("%s: %d nodes, %d edges, total weight %d", format_name, partnership_graph.number_of_nodes(),
                   partnership_graph.number_of_edges(), int(partnership_graph.size(weight="weight")))

aggregation_results = pd.DataFrame(aggregation_results)
write_table(aggregation_results, TABLES_FOLDER / "validation_aggregation_checks.csv")

2026-11-16 20:50:14,201 INFO batnet.05_networks: Test: 51 nodes, 335 edges, total weight 43080


2026-11-16 20:50:14,213 INFO batnet.05_networks: ODI: 50 nodes, 262 edges, total weight 22929


2026-11-16 20:50:14,224 INFO batnet.05_networks: T20I: 44 nodes, 222 edges, total weight 18974


### Gate

In [4]:
players_with_several_spellings = player_table[player_table["name_variants"].str.contains("|", regex=False)]
stage_log.info("players with more than one name spelling: %d", len(players_with_several_spellings))
all_checks_passed = bool(aggregation_results["passed"].all())
stage_log.info("aggregation checks passed: %d/%d - GATE %s", aggregation_results["passed"].sum(),
               len(aggregation_results), "PASSED" if all_checks_passed else "FAILED")
assert all_checks_passed, "Aggregation gate failed"

2026-11-16 20:50:14,228 INFO batnet.05_networks: players with more than one name spelling: 0


2026-11-16 20:50:14,229 INFO batnet.05_networks: aggregation checks passed: 36/36 - GATE PASSED
